# Exploration of CLIP flaws

Multimodal model can have unexpected biases. CLIP, for example, is strongly biased toward text in the text. A great example can be found [here](https://arxiv.org/pdf/2508.05430#page=8.94), Figure 6, where CLIP "sees" a doll, but is actually focused on "dollar" text, not an actual doll. Is thid a **real** problem? Considering that most of the content on web is watermarked in some way, this might be an issue. During today's lab, we will try to reproduce this phenomena on ImageNet creating artifically injected watermarks.

The coding agenda is as follow:

1. load a CLIP and stream ImageNet dataset
2. create a custom version of dataset with injected watermarks
3. iterate over dataset, compute metrics on original and injected data
4. compute simple CAV (diff means)
5. debias representation of injected data and compare metric to the original data

In [1]:
import os
import numpy as np
from PIL import ImageFont, ImageDraw, Image as PILImage
from torch.utils.data import Dataset
import torch
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

### 00 Prelimaries

code below serves as utilty to inject watermarks. You need to pass `image` and `text` to be injected, the rest of parameters can be left as default. The watermark transformation can be used via

```python
torchvision.transforms.Compose([partial(add_watermark, ...), clip_text_preprocessor])
```

In [2]:
_IMAGE_SIZE_TO_FONT_SIZE = {
    224: 36,
    384: 62,
    512: 82,
    518: 84,
}
_FONT_SIZE_RATIO = 6.22


def add_watermark(
    image: np.ndarray | PILImage.Image,
    image_size: int = 224,
    text: str = "BLUEBERRY",
    font_path: str = "/content/assets/fonts/SourceHanSerifSC-ExtraLight.otf",
    opacity: float = 0.5,
    color: tuple[int, int, int] = (255, 255, 255),
    x_pos: float = 0.01,
    y_pos: float = 0.4,
) -> PILImage.Image:
    """
    Add semi-transparent text watermark overlay to an image.

    The watermark is composited using alpha blending, preserving the original
    image with the text rendered on top at the specified opacity.

    Args:
        image: Input PIL Image or numpy array to add watermark to
        image_size: Size of input image in pixels, used for font size calculation (default: 224)
        text: Watermark text to overlay (default: WATERMARK_TEXT)
        font_path: Path to TrueType/OpenType font file (default: SourceHanSerifSC-ExtraLight.otf)
        opacity: Watermark opacity in range [0.0, 1.0] where 0.0 is fully transparent
            and 1.0 is fully opaque (default: 0.5)
        color: RGB color tuple for watermark text (default: (255, 255, 255) white)
        x_pos: Horizontal position as fraction of image width in range [0.0, 1.0] (default: 0.01)
        y_pos: Vertical position as fraction of image height in range [0.0, 1.0] (default: 0.4)

    Returns:
        PIL Image with watermark overlay applied, converted to RGB mode

    Raises:
        FileNotFoundError: If font_path doesn't exist
        ValueError: If opacity, x_pos, or y_pos are outside the valid range [0.0, 1.0]
    """
    # Validate font path
    if not os.path.exists(font_path):
        raise FileNotFoundError(f"Font file not found: {font_path}")

    # Validate range parameters
    if not 0.0 <= opacity <= 1.0:
        raise ValueError(f"Opacity must be in range [0.0, 1.0], got {opacity}")
    if not 0.0 <= x_pos <= 1.0:
        raise ValueError(f"x_pos must be in range [0.0, 1.0], got {x_pos}")
    if not 0.0 <= y_pos <= 1.0:
        raise ValueError(f"y_pos must be in range [0.0, 1.0], got {y_pos}")

    # Calculate appropriate font size
    if image_size in _IMAGE_SIZE_TO_FONT_SIZE:
        font_size = _IMAGE_SIZE_TO_FONT_SIZE[image_size]
    else:
        font_size = int(image_size / _FONT_SIZE_RATIO)

    font = ImageFont.truetype(font_path, font_size)

    # If np.array convert to PIL
    if isinstance(image, np.ndarray):
        image = PILImage.fromarray(image.astype(np.uint8), 'RGB')

    # Convert to RGBA for alpha compositing
    image_rgba = image.convert("RGBA")
    width, height = image_rgba.size

    # Create transparent overlay for watermark text
    watermark_layer = PILImage.new("RGBA", image_rgba.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(watermark_layer)

    # Draw text with specified opacity
    draw.text(
        xy=(x_pos * width, y_pos * height),
        text=text,
        fill=(*color, round(opacity * 255)),  # Cleaner tuple unpacking
        font=font,
    )

    # Composite watermark onto image and convert back to RGB
    output_img = PILImage.alpha_composite(image_rgba, watermark_layer).convert("RGB")
    return output_img

### 01 CLIP and Imagenet loading

You should already know how to load CLIP from previous labs. For ImageNet, you need to find it on HuggingFace. You can check for `imagenet-1k` for smaller volume. Note that "mini" version is still too bug to make the full experiment, so feel free to constraint experiments to 100-200 images. Use `streaming=True`, so the datasrt is not donwloaded on your PC/notebook, but is streamed from web. It is much slower, but fits our needs best.

Side note: `test` split of ImageNet does not have labels.

In [3]:
# load CLIP
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to("cpu")
processor = CLIPProcessor.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [4]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [5]:
# load ImageNet
dataset = load_dataset("ILSVRC/imagenet-1k", streaming=True, split = "train")
print(dataset)

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

IterableDataset({
    features: ['image', 'label'],
    num_shards: 294
})


### 02 Custom dataset

Because of `streaming=True`, you already have `IterableDatset` object. It already has `__getitem__` implemented, so we need to add our custom logic in `__iter__`. The logic should be as follows:

```python
    def __iter__(self):
        for example in self.dataset:
            <your code here>
            yield img, img_corrupted, label
```

Remember that `yield` should return `Tensor`, `dict[str, Tensor]`, or so. Ensure that your custom dataset returns (original image after CLIP processing, image after watermark injection and CLIP processing, original label).

Tip: datasets typically returns index of a class, not its label (e.g., `1`, not `goldfish`). You can still find label using `dataset.features["label"].int2str(IDX)`.

Tip 2: because of streaming, the biggest bottleneck is HuggingFace API, consider setting `batch_size` on `64` or more.

In [6]:
class ImageNetDataset(Dataset):
  def __init__(self, hf_dataset):
    self.dataset = hf_dataset
    self.features = hf_dataset.features

  def __iter__(self, batch_size=64):
    for example in self.dataset:
      img = example.get("image")
      label = dataset.features["label"].int2str(example.get("label"))
      img_corrupted = add_watermark(img)
      yield img, img_corrupted, label

dataset = ImageNetDataset(dataset)

### 03 Metrics computation

So far, you should have loaded CLIP and iterable dataset. Now it's time to creating **interesting** experiment. The design is up to you, but let's check my proposal below:

> - set watermark text to something like "BLUEBERRY"
> - use CLIP in zero-shot mode (comparing embedding of text and image)
> - check how many images will be predicted as "BLUEBERRY" rather than the original class
>   - e.g., compare cossim(image_watermark, "a photo of a BLUEBERRY") and cossim(image_watermark, "a photo of [ORIGINAL CLASS]")

After this step, you should tell the accuracy of the CLIP in distinguishing between BLUEBERRY and [ORIGINAL CLASS] on original data and data with watermarks (or analogous metrics in our own experiment design).

Tip: use `padding=True` when processing text in batches.

Note: I proposed BLUEBERRY for no particural reason, but keep in mind tht ImageNet contains ~all types of images so to be completly fair in this experiment we should choose something that does not occur in this dataset probably.

In [7]:
num_samples = 256
blueberry_wins_clean = 0
blueberry_wins_watermarked = 0

for i, (img, img_corrupted, true_label) in enumerate(dataset):
  if i >= num_samples:
    break

  texts = ["a photo of a blueberry", true_label]

  inputs = processor(
    text=texts,
    images=[img, img_corrupted],
    return_tensors="pt",
    padding=True
  ).to("cpu")

  with torch.no_grad():
    outputs = model(**inputs)

  probs = outputs.logits_per_image.softmax(dim=1)

  if probs[0, 0] > probs[0, 1]:
    blueberry_wins_clean += 1
  if probs[1, 0] > probs[1, 1]:
    blueberry_wins_watermarked += 1

print(f"Clean images with predicted blueberry label: {blueberry_wins_clean} / {num_samples}"
      f"({(blueberry_wins_clean/num_samples)*100:.1f}%)")
print(f"Watermarked images with predicted blueberry label: {blueberry_wins_watermarked} / {num_samples}"
      f"({(blueberry_wins_watermarked/num_samples)*100:.1f}%)")

Clean images with predicted blueberry label: 1 / 256(0.4%)
Watermarked images with predicted blueberry label: 22 / 256(8.6%)


### 04 CAV computation

In this step, you should create a CAV (e.g., DiffMeans we used in previous work) and check its detection power - i.e., `rocauc(cossim(CAV, data))`. This should be really easy!

In [8]:
# get CLIP embeddings for all 264 samples
clean_embeds = []
watermarked_embeds = []
for i, (img, img_corrupted, _) in enumerate(dataset):
  if i >= num_samples:
    break

  inputs = processor(
        text=["a", "b"], # these are dummy texts - for some reason I can't make it work without this, but it shouldn't mess with results if I understand CLIP correctly
        images=[img, img_corrupted],
        return_tensors="pt",
        padding=True
    ).to("cpu")

  with torch.no_grad():
    outputs = model(**inputs)
    embeds = outputs.image_embeds

  clean_embeds.append(embeds[0].cpu())
  watermarked_embeds.append(embeds[1].cpu())

# stack into tensors
X_clean = torch.stack(clean_embeds)
X_watermarked = torch.stack(watermarked_embeds)

# labels: 0 - no watermark, 1 - watermarked
y_clean = torch.zeros(X_clean.size(0))
y_watermarked = torch.ones(X_watermarked.size(0))

# combine into one dataset
X_all = torch.cat([X_clean, X_watermarked], dim=0)
y_all = torch.cat([y_clean, y_watermarked], dim=0)

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_all.numpy(), y_all.numpy(), test_size=0.3, stratify=y_all.numpy(), random_state=42
)

# calculate CAV
X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)

train_clean_embeds = X_train[y_train == 0]
train_watermarked_embeds = X_train[y_train == 1]

clean_mean = train_clean_embeds.mean(dim=0)
watermarked_mean = train_watermarked_embeds.mean(dim=0)
cav = watermarked_mean - clean_mean

# cossim roc-auc
similarities = F.cosine_similarity(X_test, cav.unsqueeze(0) )
roc_auc = roc_auc_score(y_test, similarities.numpy())

print(f"final ROC-AUC score: {roc_auc:.4f}")

final ROC-AUC score: 0.9344


### 05 Debiasing

Now we have everything we need to make CLIP robust to watermarks. In this step, repeat the logic from **03**, but make an ortogonalisation of data with injected watermark. Ideally, its CLIP's embedding should be similar to the original data (you can check!); however, our main goal is to just improve the accuracy of zero-shot classification.

In [9]:
def orthogonalize(X: torch.Tensor, v: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
  denom = np.dot(v, v) + eps  # (v^T v)
  projection_coeffs = X @ v / denom  # shape (N,)
  X_orth = X - np.outer(projection_coeffs, v) * 0.99  # subtract projection
  return X_orth

baseline_correct = 0
debiased_correct = 0
for i, (img, img_corrupted, true_label) in enumerate(dataset):
    if i >= num_samples:
        break

    texts = ["a photo of a blueberry", true_label]

    inputs = processor(
        text=texts,
        images=img_corrupted,
        return_tensors="pt",
        padding=True
    ).to("cpu")

    with torch.no_grad():
        outputs = model(**inputs)

        img_embed = outputs.image_embeds
        text_embeds = outputs.text_embeds

    # biased
    baseline_logits = img_embed @ text_embeds.t()
    if baseline_logits.argmax() == 1: # true label predicted
        baseline_correct += 1

    # debiased
    img_embed_orth = orthogonalize(img_embed[0], cav)
    img_embed_orth = torch.nn.functional.normalize(img_embed_orth, p=2, dim=-1)
    debiased_logits = torch.matmul(img_embed_orth, text_embeds.t())
    if debiased_logits.argmax() == 1:
        debiased_correct += 1

print(f"baseline accuracy:   {(baseline_correct / num_samples) * 100:.1f}%")
print(f"debiased accuracy:  {(debiased_correct / num_samples) * 100:.1f}%")

/tmp/ipykernel_52184/3646812955.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  denom = np.dot(v, v) + eps  # (v^T v)
/tmp/ipykernel_52184/3646812955.py:4: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  X_orth = X - np.outer(projection_coeffs, v) * 0.99  # subtract projection


baseline accuracy:   91.4%
debiased accuracy:  97.3%
